In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GATConv
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import os

# --- CONFIG ---
DATASET_FILE = "/content/drive/MyDrive/BWAF-Net-Training/Final_Aligned_Dataset.csv"
GRAPH_FILE = "/content/drive/MyDrive/BWAF-Net-Training/graph_data.pt"
OUTPUT_DIR = "/content/drive/MyDrive/BWAF-Net-Training/results_fusion_baselines/"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "best_cross_attn.pth")
BATCH_SIZE = 16
LR = 0.0005
EPOCHS = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- MODEL: Transformer + GAT + Cross-Attention Fusion ---
# (Represents EPInformer / Standard SOTA Fusion)
class CrossAttn_Net(nn.Module):
    def __init__(self, num_genes, num_tf_features, d_model=64, prior_dim=11):
        super(CrossAttn_Net, self).__init__()

        # 1. Sequence Branch (Standard Transformer)
        self.embedding = nn.Embedding(5, d_model, padding_idx=4)
        self.pos_encoder = nn.Parameter(torch.randn(1, 2000, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 2. Graph Branch (Online GAT)
        self.gat = GATConv(num_tf_features, d_model, heads=1)
        self.register_buffer('edge_index', None)
        self.register_buffer('node_features', None)

        # 3. FUSION: Multi-Head Cross Attention (The SOTA Baseline)
        # Query = Sequence, Key/Value = Graph
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=4, batch_first=True)

        # 4. Classifier
        # Input is fused(64) + priors(11)
        # We include priors in the classifier so the ONLY difference is the FUSION MECHANISM.
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model + prior_dim),
            nn.Linear(d_model + prior_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def set_graph_data(self, x, edge_index):
        self.edge_index = edge_index
        self.node_features = x

    def forward(self, seq, priors, gene_idx):
        # A. Sequence [Batch, 2000, 64]
        pad_mask = (seq == 4)
        x_seq = self.embedding(seq) + self.pos_encoder
        x_seq = self.transformer(x_seq, src_key_padding_mask=pad_mask)

        # Global Average Pooling for Seq [Batch, 1, 64]
        mask = (~pad_mask).unsqueeze(-1).float()
        h_seq = (x_seq * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        h_seq = h_seq.unsqueeze(1) # Add sequence dim for attention

        # B. Graph [Batch, 1, 64]
        all_nodes = F.elu(self.gat(self.node_features, self.edge_index))
        h_graph = all_nodes[gene_idx].unsqueeze(1)

        # C. CROSS ATTENTION FUSION
        # Query: Sequence ("What syntax do I have?")
        # Key/Value: Graph ("What context fits this syntax?")
        # Output: Context-aware Sequence features
        attn_output, _ = self.cross_attn(query=h_seq, key=h_graph, value=h_graph)
        h_fused = attn_output.squeeze(1) # [Batch, 64]

        # D. Output (Identical to BWAF)
        h_final = torch.cat([h_fused, priors], dim=1)
        return self.classifier(h_final)

# --- DATASET (Same as always) ---
class GenomicDataset(Dataset):
    def __init__(self, df, gene_to_idx):
        self.seqs = df['sequence'].values
        self.labels = df['label'].values
        self.gene_idxs = [gene_to_idx.get(g, 0) for g in df['gene_id'].values]
        prior_cols = ["TATA_Box", "CAAT_Box", "GC_Box", "BRE", "MRE", "PPE",
                      "Octamer", "Sp1", "E_Box", "RFX", "CpG_Count"]
        self.priors = np.log1p(df[prior_cols].values.astype(np.float32))
        self.dna_map = {'A':0, 'C':1, 'G':2, 'T':3, 'N':4}

    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        seq = [self.dna_map.get(s, 4) for s in self.seqs[i]]
        return (torch.tensor(seq, dtype=torch.long),
                torch.tensor(self.priors[i], dtype=torch.float32),
                torch.tensor(self.gene_idxs[i], dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.float32))

def train():
    print(f"--- TRAINING BASELINE: CROSS-ATTENTION FUSION (Device: {DEVICE}) ---")
    df = pd.read_csv(DATASET_FILE)
    graph = torch.load(GRAPH_FILE)
    gene_to_idx = graph['gene_to_idx']

    train_df = df[df['partition'] == 'train']
    val_df = df[df['partition'] == 'val']
    test_df = df[df['partition'] == 'test']

    train_loader = DataLoader(GenomicDataset(train_df, gene_to_idx), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(GenomicDataset(val_df, gene_to_idx), batch_size=BATCH_SIZE)
    test_loader = DataLoader(GenomicDataset(test_df, gene_to_idx), batch_size=BATCH_SIZE)

    # Init Cross-Attn Model
    model = CrossAttn_Net(graph['x'].shape[0], graph['x'].shape[1]).to(DEVICE)
    model.set_graph_data(graph['x'].to(DEVICE), graph['edge_index'].to(DEVICE))

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)

    best_val_loss = float('inf')

    for ep in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"Ep {ep+1}")
        for seq, prior, gidx, y in loop:
            seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(seq, prior, gidx).squeeze()
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

        # Val
        model.eval()
        val_loss = 0
        preds, targs = [], []
        with torch.no_grad():
            for seq, prior, gidx, y in val_loader:
                seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)
                logits = model(seq, prior, gidx).squeeze()
                val_loss += criterion(logits, y).item()
                preds.extend(torch.sigmoid(logits).cpu().numpy())
                targs.extend(y.cpu().numpy())

        avg_val = val_loss / len(val_loader)
        acc = accuracy_score(targs, np.array(preds) > 0.5)
        print(f"Ep {ep+1} | Val Loss: {avg_val:.4f} | Acc: {acc:.4f}")

        scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), MODEL_SAVE_PATH)

    # Final Test
    print("\n--- FINAL TEST EVALUATION (Cross-Attn) ---")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()
    preds, targs, probs = [], [], []
    with torch.no_grad():
        for seq, prior, gidx, y in tqdm(test_loader):
            seq, prior, gidx, y = seq.to(DEVICE), prior.to(DEVICE), gidx.to(DEVICE), y.to(DEVICE)
            logits = model(seq, prior, gidx).squeeze()
            probs.extend(torch.sigmoid(logits).cpu().numpy())
            targs.extend(y.cpu().numpy())

    print(f"Accuracy:  {accuracy_score(targs, np.array(probs) > 0.5):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(targs, probs):.4f}")
    print(f"F1 Score:  {f1_score(targs, np.array(probs) > 0.5):.4f}")

if __name__ == "__main__":
    train()

--- TRAINING BASELINE: CROSS-ATTENTION FUSION (Device: cuda) ---


Ep 1: 100%|██████████| 1487/1487 [08:14<00:00,  3.01it/s, loss=0.549]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Ep 1 | Val Loss: 0.5266 | Acc: 0.7394


Ep 2: 100%|██████████| 1487/1487 [08:11<00:00,  3.03it/s, loss=0.555]


Ep 2 | Val Loss: 0.5172 | Acc: 0.7464


...

Ep 14 | Val Loss: 0.4954 | Acc: 0.7610


Ep 15: 100%|██████████| 1487/1487 [08:09<00:00,  3.04it/s, loss=0.359]


Ep 15 | Val Loss: 0.4944 | Acc: 0.7626

--- FINAL TEST EVALUATION (Cross-Attn) ---


100%|██████████| 386/386 [00:39<00:00,  9.88it/s]


Accuracy:  0.7496
AUC-ROC:   0.8308
F1 Score:  0.7419
